# 01 — Triplet Parsing (CholecT50)

**Goal:** Load the CholecT50 JSON annotations, understand the binary-to-label mapping,
and parse the surgical action triplets `(instrument, verb, target)` for 3 sample videos.

**Videos:** VID01 (short), VID05 (medium), VID40 (longer)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from pathlib import Path
from src.triplet_parser import (
    load_categories,
    print_categories,
    parse_video,
    parse_multiple_videos,
    filter_valid_triplets,
    multi_label_analysis,
    video_summary,
    save_parsed_csv,
)

# Project paths
PROJECT_ROOT = Path('..').resolve()
TRIPLET_DIR = PROJECT_ROOT / 'data' / 'CholecT45' / 'triplet'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'parsed_triplets'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Triplet dir:', TRIPLET_DIR)
print('Available:', os.listdir(TRIPLET_DIR))

## 1. Inspect Category Dictionaries

Categories are embedded in each video JSON: **6 instruments, 10 verbs, 15 targets, 100 triplet classes**.

In [ ]:
categories = load_categories(TRIPLET_DIR / 'VID01.json')
print_categories(categories)

## 2. Parse a Single Video (VID01)

`parse_video()` decodes each 15-element annotation array into `(instrument, verb, target)`.
Multi-label frames (multiple simultaneous triplets) are fully expanded — one row per triplet.

In [ ]:
df_vid01_raw = parse_video(TRIPLET_DIR / 'VID01.json')
print(f'VID01 raw: {len(df_vid01_raw)} rows across {df_vid01_raw["frame"].nunique()} frames')
df_vid01_raw.head(10)

## 3. Handle Multi-Label Frames

Some frames have **no valid annotation** (triplet_id = -1). We filter these out
and recalculate per-frame counts.

In [ ]:
df_vid01 = filter_valid_triplets(df_vid01_raw)
print(f'VID01 valid: {len(df_vid01)} rows across {df_vid01["frame"].nunique()} frames')
print(f'Removed {len(df_vid01_raw) - len(df_vid01)} null entries')
df_vid01.head(10)

In [ ]:
# Multi-label analysis
stats = multi_label_analysis(df_vid01)
print('=== Multi-Label Frame Analysis (VID01) ===')
for k, v in stats.items():
    print(f'  {k}: {v}')

In [ ]:
# Show example multi-label frames
multi_frames = df_vid01[df_vid01['is_multi_label']]
if not multi_frames.empty:
    sample_frame = multi_frames['frame'].iloc[0]
    print(f'=== Example Multi-Label Frame {sample_frame} ===')
    display(df_vid01[df_vid01['frame'] == sample_frame][['frame', 'instrument', 'verb', 'target', 'triplet_label']])

## 4. Parse All 3 Sample Videos

Parse VID01, VID05, VID40 together and generate combined statistics.

In [ ]:
print('Parsing 3 sample videos...')
df_all_raw = parse_multiple_videos(TRIPLET_DIR, video_ids=['VID01', 'VID05', 'VID40'])
df_all = filter_valid_triplets(df_all_raw)

print(f'\nCombined: {len(df_all)} valid rows, {len(df_all_raw) - len(df_all)} null entries removed')
print(f'Total frames: {df_all.groupby("video")["frame"].nunique().sum()}')

In [ ]:
# Per-video summary table
summary = video_summary(df_all)
display(summary)

In [ ]:
# Multi-label analysis across all 3 videos
stats_all = multi_label_analysis(df_all)
print('=== Combined Multi-Label Analysis ===')
for k, v in stats_all.items():
    print(f'  {k}: {v}')

## 5. Label Frequency Summary

In [ ]:
print('=== Combined Label Frequencies ===')
print('\nTop 15 Triplets:')
print(df_all['triplet_label'].value_counts().head(15))
print('\nInstruments:')
print(df_all['instrument'].value_counts())
print('\nVerbs:')
print(df_all['verb'].value_counts())
print('\nTargets:')
print(df_all['target'].value_counts())

## 6. Save Parsed Triplets to CSV

In [ ]:
# Save per-video CSVs
print('Saving per-video CSVs...')
saved_files = save_parsed_csv(df_all, OUTPUT_DIR, per_video=True)

# Also save combined
print('\nSaving combined CSV...')
save_parsed_csv(df_all, OUTPUT_DIR, per_video=False)

print(f'\nAll outputs in: {OUTPUT_DIR}')
print('Files:', os.listdir(OUTPUT_DIR))

---
**Next notebook:** `02_graph_construction.ipynb` — build causal knowledge graphs from the parsed triplets.